In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import (
    DEFAULT_CUBE_SIZES,
    BereaPatchDataset,
    DigitalCorePipeline,
    FiLMRoutedUNet3D,
    PoreNetworkPermeabilityModel,
    check_required_dependencies,
)

check_required_dependencies()
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: f:\PycharmProjects\micro_ct device: cuda


In [3]:
SEG_CKPT_CANDIDATES = [
    ROOT / "models" / "film_routed_unet3d_best.pth",
    ROOT / "models" / "1epoch_full_2rocks.pth",
    ROOT / "models" / "2epoch_full_2rocks.pth",
]
GNN_CKPT = ROOT / "models" / "gnn_pnm_best.pth"

def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def looks_like_segmentation_checkpoint(checkpoint):
    state = checkpoint.get("model_state_dict", checkpoint)
    return any(key.startswith("enc1.") or key.startswith("out_conv.") for key in state)

seg_checkpoint = None
SEG_CKPT = None
for candidate in SEG_CKPT_CANDIDATES:
    if not candidate.exists():
        continue
    checkpoint = load_checkpoint(candidate)
    if looks_like_segmentation_checkpoint(checkpoint):
        SEG_CKPT = candidate
        seg_checkpoint = checkpoint
        break

if seg_checkpoint is None:
    raise FileNotFoundError("No valid segmentation checkpoint found in models/. Re-run 01_train_segmentation.ipynb.")

seg_model = FiLMRoutedUNet3D(
    base_channels=seg_checkpoint.get("base_channels", 16),
    ctx_dim=seg_checkpoint.get("ctx_dim", 64),
).to(device)
seg_model.load_state_dict(seg_checkpoint["model_state_dict"])

gnn_checkpoint = load_checkpoint(GNN_CKPT)
if gnn_checkpoint.get("checkpoint_type") not in (None, "gnn_pnm"):
    raise ValueError(f"Unexpected GNN checkpoint type: {gnn_checkpoint.get('checkpoint_type')}")
graph_model = PoreNetworkPermeabilityModel(
    node_in=gnn_checkpoint["node_in"],
    edge_in=gnn_checkpoint["edge_in"],
    hidden=gnn_checkpoint.get("hidden", 64),
    layers=gnn_checkpoint.get("layers", 3),
    mu=1.0e-3,
).to(device)
graph_model.load_state_dict(gnn_checkpoint["model_state_dict"])
print("loaded segmentation checkpoint:", SEG_CKPT)
print("loaded GNN checkpoint:", GNN_CKPT)


RuntimeError: Error(s) in loading state_dict for FiLMRoutedUNet3D:
	Missing key(s) in state_dict: "enc1.block.0.weight", "enc1.block.1.weight", "enc1.block.1.bias", "enc1.block.4.weight", "enc1.block.5.weight", "enc1.block.5.bias", "enc2.block.0.weight", "enc2.block.1.weight", "enc2.block.1.bias", "enc2.block.4.weight", "enc2.block.5.weight", "enc2.block.5.bias", "enc3.block.0.weight", "enc3.block.1.weight", "enc3.block.1.bias", "enc3.block.4.weight", "enc3.block.5.weight", "enc3.block.5.bias", "bottleneck.block.0.weight", "bottleneck.block.1.weight", "bottleneck.block.1.bias", "bottleneck.block.4.weight", "bottleneck.block.5.weight", "bottleneck.block.5.bias", "context.proj_global.weight", "context.proj_global.bias", "context.proj_intensity.weight", "context.proj_intensity.bias", "context.proj_topo.weight", "context.proj_topo.bias", "context.proj_texture.weight", "context.proj_texture.bias", "router.alpha_logits", "router.level_mlps.0.0.weight", "router.level_mlps.0.0.bias", "router.l

In [ ]:
CUBE_SIZE = 128  # change to 64 or 192 to inspect another prepared scale
VOXEL_SIZE_M = 2.25e-6
dataset = BereaPatchDataset(ROOT, split="val", cube_size=CUBE_SIZE, use_raw_gray=False, noise_types=["none"], balance=False)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
cube = batch["x"][0].numpy()
coord = batch["coord"][0].tolist()
rock = batch["rock"][0]
cube_size = int(batch["cube_size"][0])
DOMAIN_SIZE = (cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M)
print("rock:", rock, "coord:", coord, "cube:", cube.shape)


rock: BanderaBrown coord: [0, 256, 512] cube: (1, 128, 128, 128)


In [ ]:
pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    graph_model=graph_model,
    device=device,
    threshold=0.5,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)

result = pipeline.run_cube(
    cube,
    input_is_pore_mask=False,
    domain_size=DOMAIN_SIZE,
    include_ph=True,
    compute_openpnm_baseline=True,
)

k_gnn = result.permeability.k_gnn_pnm.detach().cpu().numpy()
print("network:", result.network.metadata)
print("GNN+PNM k:", k_gnn)
print("OpenPNM k:", result.permeability.k_openpnm)


network: {'num_pores': 510, 'num_throats': 713, 'node_feature_dim': 21, 'edge_feature_dim': 16, 'ph_summary': [509.0, 0.01165812462568283, 4.851777703152038e-05, 218.0, 0.0015314563643187284, 3.7617010093526915e-05], 'topology': {'percolates_z': True, 'percolates_y': True, 'percolates_x': True, 'largest_component_ratio': 1.0, 'num_components': 1, 'dead_end_ratio': 0.4, 'skeleton_length': 0.02226608683880938, 'euler_number': -202, 'bridge_fraction': 0.3899018168449402}}
GNN+PNM k: [4.5434380e-14 6.0593695e-14 7.9706403e-14]
OpenPNM k: {'kx': 1.7924787597579325e-13, 'ky': 1.0344515593679723e-13, 'kz': 8.727342245122526e-14}


In [ ]:
row = {
    "rock": rock,
    "cube_size": cube_size,
    "z": coord[0],
    "y": coord[1],
    "x": coord[2],
    "num_pores": result.network.metadata["num_pores"],
    "num_throats": result.network.metadata["num_throats"],
    "gnn_pnm_kx": float(k_gnn[0]),
    "gnn_pnm_ky": float(k_gnn[1]),
    "gnn_pnm_kz": float(k_gnn[2]),
}
if result.permeability.k_openpnm is not None:
    row.update({f"openpnm_{k}": v for k, v in result.permeability.k_openpnm.items()})

df = pd.DataFrame([row])
path = OUT_DIR / "full_pipeline_summary.csv"
df.to_csv(path, index=False)
print("saved:", path)
df


saved: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\full_pipeline_summary.csv


,rock,cube_size,z,y,x,num_pores,num_throats,gnn_pnm_kx,gnn_pnm_ky,gnn_pnm_kz,openpnm_kx,openpnm_ky,openpnm_kz
0,BanderaBrown,128,0,256,512,510,713,4.543438e-14,6.059369e-14,7.970640e-14,1.792479e-13,1.034452e-13,8.727342e-14
